# Research 01 — 训练循环零起（Training Loop from Scratch）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Papa-Panda/field-notes/blob/main/pytorch/research/01-training-loop.ipynb)

- **日期**：2026-10-02
- **定位**：Research PyTorch 轮次的底盘。这一轮考的不是会不会用 Trainer，是给你一段裸训练代码，能不能现场看出错在哪、改对。
- **环境**：纯 PyTorch，无 GPU 要求，Colab 免费版直接跑。

配套笔记：[01-training-loop.md](01-training-loop.md)


## 今日目标

1. 从零手打一个训练循环，五件事的顺序能背下来并解释为什么。
2. 验证 grad accumulation 与大 batch 的等价性（数值对比）。
3. 亲手踩三个经典 bug，看症状、定位、修复。


## 1. 一个 step 的五件事

forward → loss → zero_grad → backward → step。

PyTorch 的 autograd 默认**累积**梯度（`.grad +=`），不清零，上一步的梯度会污染这一步 —— 这是设计，不是 bug，grad accumulation 就靠它。


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# 玩具任务：y = 3x + 2 + noise
X = torch.randn(256, 1)
y = 3 * X + 2 + 0.1 * torch.randn(256, 1)

model = nn.Linear(1, 1)
opt = torch.optim.SGD(model.parameters(), lr=0.1)
lossf = nn.MSELoss()

for step in range(60):
    pred = model(X)                 # 1. forward
    loss = lossf(pred, y)           # 2. loss
    opt.zero_grad()                 # 3. zero_grad
    loss.backward()                 # 4. backward
    opt.step()                      # 5. step
    if step % 15 == 0:
        print(f'step {step:3d}  loss {loss.item():.4f}')

w, b = model.weight.item(), model.bias.item()
print(f'\n学到的参数: w={w:.3f} (真值 3), b={b:.3f} (真值 2)')


## 2. 结构化版本：train/eval + scheduler + grad accumulation

面试里更常见的形态：带 scheduler、带验证集、用 accumulation 模拟大 batch。下面验证一个关键结论：

> micro-batch 切 $k$ 份、每个 `loss/k` 后 backward、最后 step —— 梯度严格等于大 batch 一步。


In [ ]:
import copy

torch.manual_seed(0)
X = torch.randn(64, 4)
y = torch.randn(64, 1)

def one_big_step(model, opt):
    opt.zero_grad()
    loss = nn.MSELoss()(model(X), y)
    loss.backward()
    return torch.cat([p.grad.view(-1) for p in model.parameters()]).clone()

def accum_steps(model, opt, k=4):
    opt.zero_grad()
    micro = X.chunk(k), y.chunk(k)
    Xs, ys = X.chunk(k), y.chunk(k)
    for i in range(k):
        loss = nn.MSELoss()(model(Xs[i]), ys[i]) / k   # ← 除以 k 是等价性的关键
        loss.backward()
    return torch.cat([p.grad.view(-1) for p in model.parameters()]).clone()

m1, m2 = nn.Linear(4, 1), nn.Linear(4, 1)
m2.load_state_dict(copy.deepcopy(m1.state_dict()))
g_big = one_big_step(m1, torch.optim.SGD(m1.parameters(), lr=0.1))
g_acc = accum_steps(m2, torch.optim.SGD(m2.parameters(), lr=0.1))

print('大 batch 一步 vs 累积 4 步的梯度最大差:', (g_big - g_acc).abs().max().item())
print('（浮点误差量级，≈0 即等价；忘了除以 k，差会精确放大 4 倍）')


## 3. Bug 演练：三个经典坏循环

三个**故意写错**的版本，症状各不相同。每个 bug 下面直接跟修复，不用往后翻。


In [ ]:
# BUG A：忘了 zero_grad —— 梯度无限累积
torch.manual_seed(0)
model = nn.Linear(4, 1)
opt = torch.optim.SGD(model.parameters(), lr=0.05)
lossf = nn.MSELoss()
Xb = torch.randn(32, 4); yb = torch.randn(32, 1)
for step in range(5):
    loss = lossf(model(Xb), yb)
    # opt.zero_grad()  ← 漏了
    loss.backward()
    opt.step()
print('BUG A: 梯度模长（第二步起持续膨胀）', 
      [round(torch.cat([p.grad.view(-1) for p in model.parameters()]).norm().item(), 3) for _ in range(1)])


**修复 A**：每步 `opt.zero_grad()` 一次（放在 backward 前或 step 后等价，但必须每步一次；grad accumulation 时只在最后一个 micro-batch 后清）。

**为什么有效**：上一步梯度被清零后，`.grad` 只含当前 step 的梯度，更新方向恢复干净。下面把同一段代码加上 `zero_grad` 重跑，看梯度模长回到正常。

In [ ]:
# 修复版 A：补上 zero_grad
torch.manual_seed(0)
model = nn.Linear(4, 1)
opt = torch.optim.SGD(model.parameters(), lr=0.05)
lossf = nn.MSELoss()
norms = []
for step in range(5):
    loss = lossf(model(Xb), yb)
    opt.zero_grad()   # ← 补上的这一行
    loss.backward()
    opt.step()
    norms.append(round(torch.cat([p.grad.view(-1) for p in model.parameters()]).norm().item(), 3))
print('FIX A: 每步梯度模长稳定不膨胀:', norms)

In [ ]:
# BUG B：scheduler.step 放在 optimizer.step 之前 —— lr 整体错位一格
torch.manual_seed(0)
model = nn.Linear(4, 1)
opt = torch.optim.SGD(model.parameters(), lr=0.1)
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=2, gamma=0.1)
applied = []
for epoch in range(4):
    sched.step()   # ← 错位：PyTorch>=1.1 要求先 opt.step()
    applied.append(round(opt.param_groups[0]['lr'], 4))  # 优化器这步实际用的 lr
    opt.step()
print('BUG B: 每步实际用到的 lr =', applied, '（第 2 步就掉到 0.01，比计划早一格）')

**修复 B**：调换顺序 —— 先 `opt.step()` 再 `sched.step()`。

**为什么有效**：scheduler 是"按已完成的更新步数"查表的，在优化器步子迈出去之前查表，等于整个 lr 曲线提前一格；warmup 类调度上这会直接训崩。下面是正确顺序下同一个 StepLR 的曲线，注意第一格 lr 不再被跳过。

In [ ]:
# 修复版 B：先 optimizer.step 再 scheduler.step
torch.manual_seed(0)
model = nn.Linear(4, 1)
opt = torch.optim.SGD(model.parameters(), lr=0.1)
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=2, gamma=0.1)
applied = []
for epoch in range(4):
    applied.append(round(opt.param_groups[0]['lr'], 4))  # 先记录这步实际用的 lr
    opt.step()        # ← 先更新
    sched.step()      # ← 再查表调 lr
print('FIX B: 每步实际用到的 lr =', applied, '（0.1 用满两步再降，与 step_size=2 的计划一致）')

In [ ]:
# BUG C：eval 时忘 no_grad —— 显存与计算白白浪费（这里用 grad 图验证）
torch.manual_seed(0)
model = nn.Linear(4, 1)
xval = torch.randn(8, 4)
out_wrong = model(xval)                      # 忘 no_grad：建图
with torch.no_grad():
    out_right = model(xval)                  # 正确：不建图
print('BUG C: 忘 no_grad 时输出带 grad_fn =', out_wrong.grad_fn is not None,
      '| 正确版 grad_fn =', out_right.grad_fn)


**修复 C**：验证/推理一律写成

```python
model.eval()
with torch.no_grad():
    out = model(x)
model.train()   # 切回来，别只去不回
```

**为什么有效**：`no_grad` 关掉计算图构建（省显存和算力），`eval()` 关掉 dropout/BatchNorm 的训练行为 —— 上面的代码里 `out_right` 的 `grad_fn` 为 None 就是不建图的证据。两个要一起用：只用 `no_grad` 不切 eval，BN 仍会拿 batch 统计量污染结果（Case D 会展开）。

## 4. 进阶 debug case：从症状反推根因

面试给代码找 bug，考的是"症状 → 机制 → 定位"的链条。再补三个高频 case，每个都有独特的症状指纹。

**Case D：验证时忘 `model.eval()`，BN 的 running stats 被验证集污染。**
指纹：验证本身没事，但验证之后继续训练，loss 莫名变差；或者验证指标随 batch 大小漂移。机制：BN 在 train 模式下用当前 batch 的均值方差并**更新 running stats**，拿它跑验证等于用验证集改模型状态。

In [ ]:
# Case D：验证忘 model.eval() —— BN running stats 被污染
torch.manual_seed(0)
bn = nn.BatchNorm1d(8)
x = torch.randn(64, 8) * 2 + 5          # 训练数据分布
before = bn.running_mean.clone()
with torch.no_grad():
    bn(torch.randn(64, 8) * 0.1 - 5)    # 忘 eval()，用验证集（不同分布）过一遍 BN
print('Case D: running_mean 被验证集拉动的幅度 =', (bn.running_mean - before).abs().max().item())
print('→ 修复：验证前 model.eval()，验证后 model.train() 切回来，两个都不能少')

**Case E：X 与 y 配对错位（shuffle 只打乱了一边）。**
指纹：loss 卡在"随机猜"的水平纹丝不动，但代码零报错。机制：监督信号被摧毁，模型只能学输出均值。排查靠一个 sanity check：先拿 8 个样本 **overfit 到 0**，过不了这一关别谈调参。

In [ ]:
# Case E：label 配对错位 —— loss 卡在随机水平
torch.manual_seed(0)
X = torch.randn(256, 1); y_true = 3 * X + 2
y_bad = y_true[torch.randperm(256)]     # 只打乱 y，配对关系全毁

def train(y, steps=100):
    torch.manual_seed(0)
    m = nn.Linear(1, 1); o = torch.optim.SGD(m.parameters(), lr=0.1)
    for _ in range(steps):
        loss = nn.MSELoss()(m(X), y)
        o.zero_grad(); loss.backward(); o.step()
    return loss.item()

print('Case E: 配对正确时 loss =', round(train(y_true), 4), '| 配对错位时 loss =', round(train(y_bad), 2), '（≈y 的方差，学了个寂寞）')

**Case F：对 leaf 参数做 in-place 操作。**
指纹：直接抛 `RuntimeError: a leaf Variable that requires grad is being used in an in-place operation`。机制：autograd 靠 version counter 追踪张量是否被原地改过，改了就拒绝 backward（怕梯度算错）。绕法：`with torch.no_grad():` 里改，或者用 `param.data`——但后者会绕过检查、把错误藏起来，是更坏的习惯。

In [ ]:
# Case F：in-place 改 leaf 参数 —— autograd 直接拒绝
w = torch.tensor([1.0, 2.0], requires_grad=True)
try:
    w += 1.0                            # in-place，触发 version counter 保护
    print('Case F: 没报错？不应该')
except RuntimeError as e:
    print('Case F: RuntimeError →', str(e)[:80], '...')
with torch.no_grad():
    w += 1.0                            # 正确姿势：在 no_grad 里改
print('Case F 修复后: w =', w.detach().tolist(), '（optimizer.step 内部就是这么干的）')

## 小结

- 五件事的顺序：forward → loss → zero_grad → backward → step。
- accumulation 等价性的命门是 `loss/k`。
- 六个 bug、六种指纹：A 梯度膨胀、B lr 错位、C 显存漏、D 验证后训练变差（BN 污染）、E loss 卡在随机水平（配对错位）、F in-place 直接报错 —— 面试给代码找 bug，先按症状对号入座。

**下一篇**：Research 02 — DPO loss 手写 + debug（电话点名的实操形式）
